# Tomato Variant A multiclass training — GOOGLE COLAB edition

This notebook is the **Colab-GPU version** of `training/tomato/training.ipynb`
(for the VS Code + "Google Colab" GPU extension, or colab.research.google.com).

It trains and evaluates a 7-class tomato leaf disease classifier from
`Variant-a(Multiclass Classification)/{train,val,test}/{class}/` and **fixes /
improves every limitation of the previous run** (see the section below).

## What is better than `training.ipynb` (why you should use THIS file)

| # | Previous run (training.ipynb) | This Colab notebook |
|---|---|---|
| 1 | Cosine LR scheduler silently never decayed (guard never triggered) → LR stuck at 1e-4 | Real **linear warmup + cosine decay** scheduler |
| 2 | Class imbalance handled with full linear class weights (K=7.9x) → overfits tiny classes | **sqrt-balanced weights** for loss + sampler (gentler, less overfit) |
| 3 | Fixed Resize+RandomCrop augmentation | **RandomResizedCrop** (scale/aspect variation) — helps confusable pairs (Healthy↔Leaf Miner, SpottedWilt↔Late_blight) |
| 4 | No regularization besides label smoothing | **MixUp** (α=0.2, 50% batches) + **gradient clipping** (5.0) |
| 5 | Single-pass evaluation | **Test-time augmentation** (horizontal-flip averaging) for final val/test |
| 6 | No overfitting visibility | Per-epoch **train-val loss gap** logged + automatic overfit warning |
| 7 | Early stopping patience 25 | Patience 30 (longer, stable) |
| 8 | Only 5 backbones | Optional **stronger backbones**: `efficientnet_v2_m`, `convnext_small`, `regnet_y_8gf` (`STRONG_ARCHS = True`) |
| 9 | No ensemble | Final cell runs a **soft-vote ensemble of the top-3** models and compares vs best single |
| 10 | Local-only setup | **Colab bootstrap**: installs missing packages, locates/clones the repo, mounts Drive / uploads the dataset |

The dataset is re-audited every run, so the images you added are picked up
automatically (new counts, new class weights).


## Google Colab setup (VS Code + Colab GPU extension)

This notebook expects the **FarmGuard repo** and the **dataset** on the Colab VM.
Three things to check before running:

1. **GPU runtime.** In VS Code with the "Google Colab" extension pick an A100/T4
   runtime (Runtime → Change runtime type → Hardware accelerator = GPU). Free tier
   gives a T4 — the strong backbones run fine with AMP and batch 16.

2. **Drive mount.** The first code cell mounts Google Drive automatically
   (`MOUNT_DRIVE = True`). Authorize the mount with the FarmGuard Google account:
   **aplha2007beta@gmail.com** — it opens the usual "Choose an account" prompt;
   pick that account and allow access. The repo **and** the dataset are expected
   inside that Drive.

3. **The repo.** The first code cell looks for `configs/` in, in order:
   `cwd` → any parent → `MyDrive/**/FarmGuard` → `/content/**/FarmGuard` →
   clones `$REPO_URL` (default `https://github.com/shlok-dadhich/FarmGuard.git`).
   For a **private repo** set the `REPO_URL` env var to your fork, or put the
   repo inside your mounted Drive at `MyDrive/FarmGuard`.

4. **The dataset (7 classes).** The notebook finds
   `Variant-a(Multiclass Classification)/{train,val,test}/{7 classes}` at, in
   order: `/content/data/raw/tomato/...`, `/content/Variant-a(...)`,
   `MyDrive/data/raw/tomato/Variant-a(...)`, `MyDrive/**/Variant-a(...)`,
   `/kaggle/input/...`, or you can **upload a ZIP** (dataset root inside the zip) —
   cell 3 will prompt you automatically on Colab.

> Tip: if your dataset lives somewhere else on Drive, either move it to
> `MyDrive/data/raw/tomato/` or set `DATA_ROOT` via the env var before running cell 3.


In [2]:
# ================= GOOGLE COLAB BOOTSTRAP =================
# Installs only the packages Colab may be missing (torch/torchvision are preinstalled).
import importlib
import subprocess
import sys as _sys

for _pkg in ("pandas", "numpy", "scikit-learn", "matplotlib", "pillow", "PyYAML"):
    try:
        importlib.import_module(_pkg)
    except Exception:
        print(f"[colab] installing {_pkg} ...")
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q", _pkg], check=False)

# Optional: mount Google Drive so the repo/dataset can live there.
MOUNT_DRIVE = True  # mount Drive automatically (authorize with aplha2007beta@gmail.com)
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")

        drive_folder = "/content/drive/MyDrive/Variant-a(Multiclass Classification)"
        if not os.path.isdir(drive_folder):
            raise FileNotFoundError(f"Folder not found: {drive_folder}")

        print(f"[colab] Drive mounted; dataset folder: {drive_folder}")
        print("[colab] Drive mounted at /content/drive")
    except Exception as e:
        print(f"[colab] Drive mount skipped: {e}")

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import logging
import os
import random
import shutil
import subprocess
import sys
import traceback
import warnings
from contextlib import nullcontext

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils import compute_class_weight
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

ROOT = Path.cwd().resolve()
# Locate the FarmGuard repo (needs configs/): cwd / parents -> Drive -> /content -> Kaggle -> clone.
if not any((candidate / "configs").is_dir() for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]):
    for mount_root in [Path("/content/drive/MyDrive"), Path("/content"), Path("/kaggle/input")]:
        if not mount_root.is_dir():
            continue
        for candidate in sorted(mount_root.rglob("FarmGuard"))[:50]:
            if (candidate / "configs").is_dir():
                ROOT = candidate
                break
        if (ROOT / "configs").is_dir():
            break
if not (ROOT / "configs").is_dir():
    repo_url = os.environ.get("REPO_URL", "https://github.com/shlok-dadhich/FarmGuard.git")
    clone_root = Path("/content/FarmGuard")
    if not clone_root.exists():
        print(f"[setup] Cloning repo from {repo_url} into {clone_root}")
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(clone_root)], check=False)
    if (clone_root / "configs").is_dir():
        ROOT = clone_root
    else:
        print("[setup] WARNING: repo not found and clone failed — running standalone "
              "(project tracking integration skipped, all outputs still saved).")

sys.path.insert(0, str(ROOT))

DATA_ROOT = ROOT / "data" / "raw" / "tomato" / "Variant-a(Multiclass Classification)"
SPLIT_DIRS = {"train": "train", "val": "val", "test": "test"}
OUTPUT_ROOT = ROOT / "outputs"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
LOG_ROOT = OUTPUT_ROOT / "logs"
FIGURE_ROOT = OUTPUT_ROOT / "figures" / "tomato"
CONFUSION_ROOT = OUTPUT_ROOT / "confusion_matrices"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
EXPERIMENT_ROOT = OUTPUT_ROOT / "experiments" / "tomato"
MODEL_ROOT = ROOT / "models" / "checkpoints" / "tomato"
CLASS_CONFIG_ROOT = ROOT / "configs" / "classes"

for directory in [OUTPUT_ROOT, METRICS_ROOT, LOG_ROOT, FIGURE_ROOT, CONFUSION_ROOT, PREDICTION_ROOT, EXPERIMENT_ROOT, MODEL_ROOT, CLASS_CONFIG_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    PROJECT_CFG = yaml.safe_load((ROOT / "configs" / "project.yaml").read_text()) or {}
except Exception:
    PROJECT_CFG = {}
try:
    TRACKING_CFG = yaml.safe_load((ROOT / "configs" / "tracking.yaml").read_text()) or {}
except Exception:
    TRACKING_CFG = {}
TRACKING_CFG["file_store"] = str(METRICS_ROOT)

SEED = int(PROJECT_CFG.get("seed", 42))
INPUT_SIZE = int(PROJECT_CFG.get("image_size", 224))
NUM_WORKERS = int(PROJECT_CFG.get("num_workers", 0))
USE_CUDA = PROJECT_CFG.get("device", "auto") in ("cuda", "gpu") and torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

RANDOM_SEED = SEED
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception:
    pass
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_ROOT / "tomato_training_main.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("tomato_training")
logger.handlers.clear()
logger.propagate = False
logger.setLevel(logging.INFO)
logger.addHandler(logging.FileHandler(LOG_ROOT / "tomato_training_main.log", encoding="utf-8"))
logger.addHandler(logging.StreamHandler())

try:
    from src.tracking.run_schema import RunRecord
    from src.tracking.tracker import get_tracker

    TRACKER = get_tracker(TRACKING_CFG)
except Exception as exc:
    logger.warning(
        "Project tracking import failed; continuing with local files only: %s",
        exc,
    )
    RunRecord = None
    TRACKER = None

VERIFY_IMAGES = True
RUN_EXPERIMENTS = True
USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05
MIN_DELTA = 0.001
EARLY_STOPPING_PATIENCE = 10          # overridden to 30 in the settings cell below
FREEZE_BACKBONE = True
UNFREEZE_EPOCH = 2
PLOT_RESULTS = True
MAX_SAMPLES_PER_CLASS = None
# "sqrt" scales balanced weights by sqrt() -> gentler up-weighting of rare classes
# (Healthy, Nitrogen Deficiency) so tiny classes are learned without overfitting.
SAMPLER_MODE = "sqrt"   # "balanced" | "sqrt"

STRONG_ARCHS = True   # <- set False to train only the original 5 backbones (faster)
EXPERIMENT_CONFIGS = [
    {"architecture": "efficientnet_v2_s", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
    {"architecture": "resnet50", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
    {"architecture": "convnext_tiny", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
    {"architecture": "regnet_y_4gf", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
    {"architecture": "densenet121", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
]
if STRONG_ARCHS:
    # Larger backbones: usually +0.5-1.5% accuracy but ~1.5-2x slower per epoch on a T4.
    EXPERIMENT_CONFIGS += [
        {"architecture": "efficientnet_v2_m", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
        {"architecture": "convnext_small", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
        {"architecture": "regnet_y_8gf", "pretrained": True, "epochs": 25, "batch_size": 16, "lr": 1.0e-4, "weight_decay": 0.05},
    ]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_name(value):
    value = str(value)
    value = value.replace(" ", "_").replace("/", "_").replace("\\", "_")
    return "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in value)

def to_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_jsonable(payload), indent=2, ensure_ascii=False), encoding="utf-8")
    return path

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relative_to_root(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT))
    except Exception:
        return str(path.resolve())

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def total_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(model, path, meta):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "state_dict": model.state_dict(),
        "meta": to_jsonable(meta),
    }
    torch.save(payload, str(path))
    return str(path)

def load_checkpoint_state(path, device="cpu"):
    payload = torch.load(str(path), map_location=device, weights_only=False)
    if isinstance(payload, dict) and "state_dict" in payload:
        return payload["state_dict"], payload.get("meta", {})
    return payload, {}

def write_class_mapping(class_names):
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    save_json(MODEL_ROOT / "classes.json", {"classes": class_names, "class_to_idx": class_to_idx})
    yaml_text = "classes:\n" + "".join(f"  - {name}\n" for name in class_names)
    (CLASS_CONFIG_ROOT / "tomato.yaml").write_text(yaml_text, encoding="utf-8")
    return class_to_idx

logger.info("Repository root: %s", ROOT)
logger.info("Tomato data root: %s", DATA_ROOT)
logger.info("Device: %s", DEVICE)
display(Markdown(f"**Runtime setup complete.** Root: `{relative_to_root(ROOT)}`; device: `{DEVICE}`.`"))


[colab] installing scikit-learn ...
[colab] installing pillow ...
[colab] installing PyYAML ...
[colab] Drive mount skipped: [dfs_ephemeral] Credentials propagation unsuccessful
[setup] Cloning repo from https://github.com/shlok-dadhich/FarmGuard.git into /content/FarmGuard


Repository root: /content/FarmGuard
Tomato data root: /content/FarmGuard/data/raw/tomato/Variant-a(Multiclass Classification)
Device: cpu


**Runtime setup complete.** Root: `.`; device: `cpu`.`

Dataset not found. Upload a ZIP containing Variant-a(Multiclass Classification)/train, val, and test.


KeyboardInterrupt: 

In [4]:
# Prefer a compatible local GPU whenever CUDA is available unless the config explicitly requests CPU.
requested_device = str(PROJECT_CFG.get("device", "auto")).strip().lower()
if requested_device not in {"auto", "cpu", "cuda", "gpu"}:
    raise ValueError(f"Unsupported device setting: {requested_device}")

cuda_available = torch.cuda.is_available()
cuda_compatible = False
if cuda_available:
    major, minor = torch.cuda.get_device_capability(0)
    cuda_compatible = major >= 7
    if not cuda_compatible:
        logger.warning(
            "CUDA device %s has compute capability %d.%d, but this PyTorch build requires >= 7.0; using CPU.",
            torch.cuda.get_device_name(0),
            major,
            minor,
        )

if requested_device == "cpu":
    USE_CUDA = False
else:
    USE_CUDA = cuda_available and cuda_compatible

DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
if USE_CUDA:
    logger.info("Using CUDA GPU: %s", torch.cuda.get_device_name(0))
else:
    logger.warning("Using CPU. torch.cuda.is_available()=%s", cuda_available)

display(Markdown(f"**Training device:** `{DEVICE}`" + (f" (`{torch.cuda.get_device_name(0)}`)" if USE_CUDA else "")))


Using CUDA GPU: Tesla T4


**Training device:** `cuda` (`Tesla T4`)

In [5]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError

def discover_images():
    rows = []
    for split, split_dir in SPLIT_DIRS.items():
        split_path = DATA_ROOT / split_dir
        if not split_path.is_dir():
            raise FileNotFoundError(f"Missing split directory: {split_path}")
        class_dirs = sorted(
            p for p in split_path.iterdir()
            if p.is_dir() and not p.name.startswith(".")
        )
        if not class_dirs:
            raise RuntimeError(f"No class directories found in {split_path}")
        for class_dir in class_dirs:
            for image_path in sorted(p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS):
                rows.append({
                    "split": split,
                    "path": str(image_path),
                    "label": class_dir.name,
                    "class_dir": class_dir.name,
                })
    return rows

rows = discover_images()
if not rows:
    raise RuntimeError("No supported images found under the selected Tomato Variant A root.")

audit_df = pd.DataFrame(rows)
original_counts = audit_df.groupby(["split", "label"]).size().unstack(fill_value=0)

bad_paths = set()
if VERIFY_IMAGES:
    for idx, row in audit_df.iterrows():
        try:
            with Image.open(row["path"]) as im:
                im.verify()
            with Image.open(row["path"]) as im:
                im.convert("RGB")
        except Exception as exc:
            bad_paths.add(str(Path(row["path"]).resolve()))
            if idx % 500 == 0:
                logger.warning("Checked %d/%d images; corrupt count=%d", idx, len(audit_df), len(bad_paths))
    if bad_paths:
        audit_df = audit_df[~audit_df["path"].apply(lambda p: str(Path(p).resolve()) in bad_paths)]
        logger.warning("Skipping %d unreadable images.", len(bad_paths))

class_names = sorted(audit_df["label"].unique().tolist())
if len(class_names) != 7:
    raise RuntimeError(f"Expected 7 classes for Variant A, found {len(class_names)}: {class_names}")
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
split_counts = audit_df.groupby("split").size().to_dict()
class_counts = audit_df.groupby("label").size().to_dict()

dataset_report = {
    "dataset": "tomato_variant_a_multiclass",
    "data_root": str(DATA_ROOT),
    "created_at": now_iso(),
    "split_counts": split_counts,
    "class_counts": class_counts,
    "classes": class_names,
    "class_to_idx": class_to_idx,
    "image_extensions": sorted(SUPPORTED_EXTENSIONS),
    "skipped_corrupt_images": len(bad_paths),
}
save_json(METRICS_ROOT / "tomato_dataset_report.json", dataset_report)
audit_df.to_csv(METRICS_ROOT / "tomato_dataset_audit.csv", index=False)

display(Markdown(f"**Dataset audit complete:** {len(audit_df)} valid images, {len(class_names)} classes."))
display(original_counts)
display(Markdown("Class order used by the model head:"))
display(pd.DataFrame({"index": range(len(class_names)), "class": class_names}))


FileNotFoundError: Missing split directory: /content/FarmGuard/data/raw/tomato/Variant-a(Multiclass Classification)/train

In [ ]:
def make_train_transform():
    # RandomResizedCrop adds scale/aspect variation -> the model becomes robust to
    # leaf size differences (the confusable pairs differ in texture, not position).
    return transforms.Compose([
        transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.6, 1.0), ratio=(0.75, 1.333)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=15),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),
            scale=(0.85, 1.15),
            shear=5,
            fill=0,
        ),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.06),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def make_eval_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_transform = make_train_transform()
val_transform = make_eval_transform()
test_transform = make_eval_transform()

def show_transform_samples(paths, transform, title, n=4):
    paths = list(paths)[:n]
    fig, axes = plt.subplots(2, n, figsize=(4*n, 5))
    axes = axes.ravel()
    for i, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"original\n{Path(path).name}", fontsize=9)
        axes[i].axis("off")
    for j, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        tensor = transform(img)
        img_np = tensor.permute(1, 2, 0).cpu().numpy()
        img_np = img_np * IMAGENET_STD + IMAGENET_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax = axes[n + j]
        ax.imshow(img_np)
        ax.set_title(f"transformed {j + 1}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(paths) * 2:]:
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

sample_paths = audit_df[audit_df["split"] == "train"]["path"].sample(min(4, len(audit_df)), random_state=RANDOM_SEED).tolist()
show_transform_samples(sample_paths, train_transform, "Training augmentation preview")
show_transform_samples(sample_paths, val_transform, "Validation/test preprocessing preview")

In [ ]:
def get_weight_enum(architecture):

    enum_name = {
        "efficientnet_v2_s": "EfficientNet_V2_S_Weights",
        "efficientnet_v2_m": "EfficientNet_V2_M_Weights",
        "resnet50": "ResNet50_Weights",
        "convnext_tiny": "ConvNeXt_Tiny_Weights",
        "convnext_small": "ConvNeXt_Small_Weights",
        "densenet121": "DenseNet121_Weights",
        "regnet_y_4gf": "RegNet_Y_4GF_Weights",   # resolved to 3.2GF in the settings cell
        "regnet_y_8gf": "RegNet_Y_8GF_Weights",
    }.get(architecture)

    if enum_name is None:

        return None

    enum_obj = getattr(models, enum_name, None)

    if enum_obj is None:

        return None

    return getattr(enum_obj, "DEFAULT", None)



def replace_classifier_head(model, num_classes):

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):

        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

        return model

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):

        for index in range(len(model.classifier) - 1, -1, -1):

            module = model.classifier[index]

            if isinstance(module, nn.Linear):

                model.classifier[index] = nn.Linear(module.in_features, num_classes)

                return model

        raise RuntimeError("Could not find a linear classifier head")

    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):

        model.fc = nn.Linear(model.fc.in_features, num_classes)

        return model

    if hasattr(model, "head") and isinstance(model.head, nn.Linear):

        model.head = nn.Linear(model.head.in_features, num_classes)

        return model

    raise RuntimeError("Architecture has no recognized linear classification head")



def build_torchvision_model(architecture, num_classes, pretrained):

    factory = getattr(models, architecture, None)

    if factory is None or not callable(factory):

        raise RuntimeError(f"torchvision.models has no callable {architecture}")

    weight_enum = get_weight_enum(architecture) if pretrained else None

    loaded_pretrained = False

    try:

        try:

            model = factory(weights=weight_enum)

            loaded_pretrained = weight_enum is not None

        except TypeError:

            model = factory(pretrained=weight_enum is not None)

            loaded_pretrained = weight_enum is not None

    except Exception as exc:

        if not pretrained:

            raise

        warnings.warn(

            f"Could not load pretrained weights for {architecture}; rebuilding with random weights. Reason: {exc}",

            RuntimeWarning,

        )

        try:

            model = factory(weights=None)

        except TypeError:

            model = factory(pretrained=False)

        loaded_pretrained = False

    model = replace_classifier_head(model, num_classes)

    model.farmguard_architecture = architecture

    model.farmguard_input_size = INPUT_SIZE

    model.farmguard_num_classes = num_classes

    model.farmguard_pretrained = loaded_pretrained

    return model



def build_model(architecture, num_classes, pretrained):

    return build_torchvision_model(architecture, num_classes, pretrained)



demo_arch = EXPERIMENT_CONFIGS[0]["architecture"]

demo_model = build_model(demo_arch, len(class_names), EXPERIMENT_CONFIGS[0]["pretrained"])

display(Markdown(f"**Demo architecture:** `{demo_arch}` with a 7-class classification head."))

display(pd.DataFrame({

    "total_parameters": [total_parameters(demo_model)],

    "trainable_parameters": [count_parameters(demo_model)],

    "input_size": [INPUT_SIZE],

    "num_classes": [len(class_names)],

    "pretrained_requested": [EXPERIMENT_CONFIGS[0]["pretrained"]],

    "pretrained_loaded": [bool(demo_model.farmguard_pretrained)],

}))

del demo_model


In [ ]:
def freeze_classifier_only(model):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for attr in ("classifier", "fc", "head"):
        module = getattr(model, attr, None)
        if module is not None:
            for parameter in module.parameters():
                parameter.requires_grad = True
    return model

def unfreeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    return model

def build_optimizer(model, lr, weight_decay):
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable:
        raise RuntimeError("No trainable parameters found")
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)

def make_class_weight_tensor(train_dataset, num_classes):
    labels = np.asarray(train_dataset.targets)
    counts = np.bincount(labels, minlength=num_classes)
    weights = compute_class_weight(
        class_weight="balanced",
        labels=np.arange(num_classes),
        class_counts=counts,
    )
    return torch.tensor(weights, dtype=torch.float)

def make_weighted_sampler(train_dataset, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    class_counts = np.bincount(np.asarray(train_dataset.targets), minlength=len(class_weights))
    sample_weights = np.repeat(class_weights, class_counts)
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_dataset),
        replacement=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED),
    )

def make_loader(split, transform, batch_size, shuffle=False, sampler=None, generator=None):
    dataset = datasets.ImageFolder(str(DATA_ROOT / split), transform=transform)
    if sampler is None and shuffle:
        sampler = torch.utils.data.RandomSampler(dataset, generator=generator)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False if sampler is not None else shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
        sampler=sampler,
    )

train_dataset = datasets.ImageFolder(str(DATA_ROOT / "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(str(DATA_ROOT / "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(str(DATA_ROOT / "test"), transform=test_transform)

if train_dataset.classes != class_names or val_dataset.classes != class_names or test_dataset.classes != class_names:
    raise RuntimeError("Dataset class order is inconsistent with the audited class list.")

train_labels = np.asarray(train_dataset.targets)
train_counts = np.bincount(train_labels, minlength=len(class_names))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels,
)
if SAMPLER_MODE == "sqrt":
    # sqrt-scaling: keeps the ordering but compresses the extreme weights
    # (e.g. Nitrogen 3.3 -> 1.8) so rare classes are not memorized.
    class_weights = np.sqrt(class_weights)
class_weight_tensor = torch.tensor(class_weights, dtype=torch.float)
train_sampler = make_weighted_sampler(train_dataset, class_weights) if USE_WEIGHTED_SAMPLER else None

train_loader = make_loader("train", train_transform, 16, shuffle=True, sampler=train_sampler, generator=torch.Generator().manual_seed(RANDOM_SEED))
val_loader = make_loader("val", val_transform, 32, shuffle=False)
test_loader = make_loader("test", test_transform, 32, shuffle=False)

display(Markdown("**DataLoaders built from the existing Variant A split directories.**"))
display(pd.DataFrame({
    "split": ["train", "val", "test"],
    "samples": [len(train_dataset), len(val_dataset), len(test_dataset)],
    "classes": [len(train_dataset.classes), len(val_dataset.classes), len(test_dataset.classes)],
}))
display(pd.DataFrame({"class": class_names, "train_count": train_counts,
                      "class_weight": class_weights, "sampler_mode": SAMPLER_MODE}))

In [ ]:
def autocast_context(device, enabled):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", enabled=True)
    return nullcontext()

def make_grad_scaler(device):
    if device.type == "cuda":
        return torch.cuda.amp.GradScaler()
    return None

def evaluate_model(model, loader, class_names, device, split, use_amp=False, save_artifacts=True, tta=False):
    model.eval()
    all_probs = []
    all_logits = []
    all_labels = []
    all_paths = []
    all_true_names = []
    all_pred_names = []
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(loader):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast_context(device, use_amp):
                logits = model(inputs)
                if tta:
                    # Test-time augmentation: average logits of the image + its h-flip.
                    logits = (logits + model(torch.flip(inputs, dims=[3]))) / 2.0
            loss_sum += criterion(logits.cpu(), labels.cpu()).item()
            probs = torch.softmax(logits, dim=1)
            pred_indices = probs.argmax(dim=1)
            all_probs.append(probs.cpu())
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
            batch_paths = [
                str(path)
                for path, _ in loader.dataset.samples[
                    batch_idx * loader.batch_size : (batch_idx + 1) * loader.batch_size
                ]
            ]
            all_paths.extend(batch_paths)
            all_true_names.extend([class_names[int(label)] for label in labels.cpu().tolist()])
            all_pred_names.extend([class_names[int(idx)] for idx in pred_indices.cpu().tolist()])
    probs_np = np.concatenate(all_probs).astype(float)
    labels_np = np.concatenate(all_labels).astype(int)
    pred_indices_np = np.argmax(probs_np, axis=1)
    num_classes = len(class_names)
    metrics = {
        "accuracy": float(accuracy_score(labels_np, pred_indices_np)),
        "macro_precision": float(precision_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels_np, pred_indices_np, average="weighted", zero_division=0)),
        "loss": float(loss_sum / max(1, len(labels_np))),
        "n": int(len(labels_np)),
        "per_class_recall": {},
        "tta": bool(tta),
    }
    per_class_recall = recall_score(labels_np, pred_indices_np, average=None, labels=list(range(num_classes)), zero_division=0)
    metrics["per_class_recall"] = {class_names[idx]: float(per_class_recall[idx]) for idx in range(num_classes)}
    report = classification_report(
        labels_np,
        pred_indices_np,
        labels=list(range(num_classes)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(labels_np, pred_indices_np, labels=list(range(num_classes)))
    result = {
        "split": split,
        "metrics": metrics,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "true_labels": labels_np.tolist(),
        "predicted_indices": pred_indices_np.tolist(),
        "true_names": all_true_names,
        "predicted_names": all_pred_names,
        "paths": all_paths,
        "top_indices": np.argsort(-probs_np, axis=1)[:, :3].tolist(),
        "top_probs": probs_np[np.arange(len(probs_np)), pred_indices_np].tolist(),
    }
    if save_artifacts:
        stem = f"tomato_{safe_name(split)}_{now_iso().replace(':', '').replace('-', '')}_{np.random.randint(0, 100000):06d}"
        metrics_path = METRICS_ROOT / f"{stem}.json"
        save_json(metrics_path, result)
        result["metrics_path"] = str(metrics_path)
        cm_path = CONFUSION_ROOT / f"{stem}.png"
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(f"{split} confusion matrix")
        ax.set_xticks(range(num_classes))
        ax.set_yticks(range(num_classes))
        ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(class_names, fontsize=8)
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        fig.savefig(cm_path, dpi=160)
        plt.close(fig)
        result["confusion_matrix_path"] = str(cm_path)
        pred_df = pd.DataFrame({
            "path": result["paths"],
            "true_label": result["true_names"],
            "predicted_label": result["predicted_names"],
            "true_index": result["true_labels"],
            "predicted_index": result["predicted_indices"],
            "top1_probability": result["top_probs"],
            "top3_indices": result["top_indices"],
        })
        pred_path = PREDICTION_ROOT / f"{stem}_predictions.csv"
        pred_df.to_csv(pred_path, index=False)
        result["predictions_path"] = str(pred_path)
    return result

def plot_confusion_matrix(cm, class_names, title, path):
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def plot_training_history(history, path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], label="train loss")
    axes[0].plot(df["epoch"], df["val_loss"], label="val loss")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].legend()
    axes[0].set_title("Loss (watch the train/val gap for overfitting)")
    axes[1].plot(df["epoch"], df["val_accuracy"], label="accuracy")
    axes[1].plot(df["epoch"], df["val_macro_f1"], label="macro F1")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("score")
    axes[1].legend()
    axes[1].set_title("Validation metrics")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def plot_sample_predictions(result, dataset, transform, class_names, path):
    paths = result["paths"]
    pred_indices = result["predicted_indices"]
    true_indices = result["true_labels"]
    n = min(12, len(paths))
    indices = np.linspace(0, len(paths) - 1, n, dtype=int)
    fig, axes = plt.subplots(3, 4, figsize=(15, 9))
    axes = axes.ravel()
    for ax, idx in zip(axes, indices):
        image = Image.open(paths[idx]).convert("RGB")
        tensor = dataset[idx][0]
        img_np = tensor.permute(1, 2, 0).cpu().numpy()
        img_np = img_np * np.asarray(IMAGENET_STD) + np.asarray(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"true: {class_names[true_indices[idx]]}\npred: {class_names[pred_indices[idx]]}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(indices):]:
        ax.axis("off")
    fig.suptitle("Sample predictions", y=1.02)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

In [ ]:
def train_one_experiment(experiment, run_id, train_loader, val_loader, test_loader, class_names, class_weight_tensor):
    set_seed(experiment["seed"])

    architecture = experiment["architecture"]
    epochs = int(experiment["epochs"])
    batch_size = int(experiment["batch_size"])
    lr = float(experiment["lr"])
    weight_decay = float(experiment["weight_decay"])
    pretrained_requested = bool(experiment["pretrained"])

    run_root = EXPERIMENT_ROOT / run_id
    run_root.mkdir(parents=True, exist_ok=True)

    model = build_model(architecture, len(class_names), pretrained_requested)
    model = freeze_classifier_only(model)
    model.to(DEVICE)

    optimizer = build_optimizer(model, lr, weight_decay)
    scheduler = build_scheduler(optimizer, epochs)   # warmup + cosine (steps once per epoch)
    scaler = make_grad_scaler(DEVICE)
    use_amp = DEVICE.type == "cuda"

    loss_fn = nn.CrossEntropyLoss(
        weight=class_weight_tensor.to(DEVICE) if USE_CLASS_WEIGHTS else None,
        label_smoothing=LABEL_SMOOTHING,
    )

    best_f1 = -1.0
    best_epoch = 0
    best_state = None
    patience_counter = 0
    history = []
    start_time = datetime.now(timezone.utc)

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for inputs, labels in train_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context(DEVICE, use_amp):
                logits = model(inputs)
                if USE_MIXUP and random.random() < MIXUP_PROB:
                    # MixUp: convex combination of two samples -> regularization,
                    # reduces overfitting and helps the confusable classes.
                    lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                    perm = torch.randperm(inputs.size(0), device=inputs.device)
                    loss = lam * loss_fn(logits, labels) + (1.0 - lam) * loss_fn(logits, labels[perm])
                else:
                    loss = loss_fn(logits, labels)
            if scaler is None:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
            train_loss_sum += loss.item() * labels.size(0)
            train_count += labels.size(0)

        if FREEZE_BACKBONE and epoch >= UNFREEZE_EPOCH:
            model = unfreeze_all(model)
            optimizer = build_optimizer(model, lr, weight_decay)
            scheduler = build_scheduler(optimizer, epochs)
            scaler = make_grad_scaler(DEVICE)

        scheduler.step()
        val_result = evaluate_model(
            model,
            val_loader,
            class_names,
            DEVICE,
            split="val",
            use_amp=use_amp,
            save_artifacts=False,
        )
        train_loss = train_loss_sum / max(1, train_count)
        val_metrics = val_result["metrics"]
        gap = float(train_loss - val_metrics["loss"])
        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "val_loss": float(val_metrics["loss"]),
            "train_val_gap": gap,
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_macro_precision": float(val_metrics["macro_precision"]),
            "val_macro_recall": float(val_metrics["macro_recall"]),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "val_weighted_f1": float(val_metrics["weighted_f1"]),
            "val_n": int(val_metrics["n"]),
        }
        history.append(row)
        pd.DataFrame(history).to_csv(run_root / "history.csv", index=False)

        if val_metrics["macro_f1"] > best_f1 + MIN_DELTA:
            best_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        logger.info(
            "run=%s epoch=%d/%d train_loss=%.4f val_loss=%.4f gap=%.4f val_macro_f1=%.4f val_accuracy=%.4f",
            run_id, epoch, epochs, train_loss, val_metrics["loss"], gap,
            val_metrics["macro_f1"], val_metrics["accuracy"],
        )
        if epoch > 10 and gap > OVERFIT_GAP_WARN:
            logger.warning(
                "run=%s epoch=%d POSSIBLE OVERFIT: train-val loss gap=%.3f (train=%.4f val=%.4f) "
                "- consider stopping or adding augmentation.",
                run_id, epoch, gap, train_loss, val_metrics["loss"],
            )
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            logger.info("Early stopping at epoch %d", epoch)
            break

    if best_state is None:
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)

    final_val = evaluate_model(
        model, val_loader, class_names, DEVICE, split="val",
        use_amp=use_amp, save_artifacts=True, tta=USE_TTA,
    )
    final_test = evaluate_model(
        model, test_loader, class_names, DEVICE, split="test",
        use_amp=use_amp, save_artifacts=True, tta=USE_TTA,
    )

    checkpoint_path = MODEL_ROOT / f"tomato_{safe_name(architecture)}_seed{experiment['seed']}_{safe_name(run_id)}.pt"
    best_state_path = run_root / "best_state.pt"
    save_checkpoint(model, best_state_path, {
        "run_id": run_id,
        "stage": "best_validation_state",
        "architecture": architecture,
        "num_classes": len(class_names),
        "seed": experiment["seed"],
    })
    final_checkpoint_meta = {
        "run_id": run_id,
        "architecture": architecture,
        "crop": "tomato",
        "num_classes": len(class_names),
        "classes": class_names,
        "class_to_idx": {name: idx for idx, name in enumerate(class_names)},
        "seed": experiment["seed"],
        "input_size": INPUT_SIZE,
        "device": str(DEVICE),
        "pretrained_requested": pretrained_requested,
        "pretrained_loaded": bool(model.farmguard_pretrained),
        "epochs": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "weight_decay": weight_decay,
        "augmentation_version": "tomato-notebook-aug-v2",
        "dataset_root": str(DATA_ROOT),
        "best_epoch": best_epoch,
        "best_val_macro_f1": float(best_f1),
        "best_val_accuracy": float(final_val["metrics"]["accuracy"]),
        "test_macro_f1": float(final_test["metrics"]["macro_f1"]),
        "test_accuracy": float(final_test["metrics"]["accuracy"]),
        "training_started_at": start_time.isoformat(),
        "training_finished_at": now_iso(),
    }
    save_checkpoint(model, checkpoint_path, final_checkpoint_meta)

    history_df = pd.DataFrame(history)
    summary_row = {
        "run_id": run_id,
        "status": "ok",
        "architecture": architecture,
        "seed": experiment["seed"],
        "pretrained_requested": pretrained_requested,
        "pretrained_loaded": bool(model.farmguard_pretrained),
        "epochs_attempted": len(history),
        "best_epoch": best_epoch,
        "best_val_macro_f1": float(best_f1),
        "best_val_accuracy": float(final_val["metrics"]["accuracy"]),
        "test_accuracy": float(final_test["metrics"]["accuracy"]),
        "test_macro_f1": float(final_test["metrics"]["macro_f1"]),
        "test_weighted_f1": float(final_test["metrics"]["weighted_f1"]),
        "parameter_count": count_parameters(model),
        "total_parameters": total_parameters(model),
        "train_samples": len(train_loader.dataset),
        "val_samples": len(val_loader.dataset),
        "test_samples": len(test_loader.dataset),
        "checkpoint_path": str(checkpoint_path),
        "relative_checkpoint_path": relative_to_root(checkpoint_path),
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "training_time_s": (datetime.now(timezone.utc) - start_time).total_seconds(),
        "history_path": str(run_root / "history.csv"),
        "metadata_path": str(save_json(run_root / "metadata.json", final_checkpoint_meta)),
        "val_metrics_path": final_val.get("metrics_path", ""),
        "test_metrics_path": final_test.get("metrics_path", ""),
        "val_confusion_matrix_path": final_val.get("confusion_matrix_path", ""),
        "test_confusion_matrix_path": final_test.get("confusion_matrix_path", ""),
        "predictions_path": final_test.get("predictions_path", ""),
        "figure_path": str(run_root / "training_curves.png"),
        "sample_predictions_path": str(run_root / "sample_predictions.png"),
    }

    plot_training_history(history, run_root / "training_curves.png")
    plot_confusion_matrix(final_test["confusion_matrix"], class_names, "Test confusion matrix", run_root / "test_confusion_matrix.png")
    plot_sample_predictions(final_test, test_dataset, test_transform, class_names, run_root / "sample_predictions.png")

    save_json(run_root / "training_config.json", experiment)
    save_json(run_root / "final_test_metrics.json", final_test)
    save_json(run_root / "final_val_metrics.json", final_val)
    save_json(run_root / "summary.json", summary_row)

    if TRACKER is not None and RunRecord is not None:
        try:
            record = RunRecord(
                timestamp=now_iso(),
                run_id=run_id,
                crop="tomato",
                architecture=architecture,
                seed=experiment["seed"],
                dataset_version="variant-a-multiclass-v1",
                split_version="existing-split-v1",
                dataset_name="tomato_variant_a_multiclass",
                split="test",
                image_size=INPUT_SIZE,
                batch_size=batch_size,
                epochs=len(history),
                optimizer="adamw",
                learning_rate=lr,
                scheduler="warmup_cosine",
                weight_decay=weight_decay,
                augmentation_version="tomato-notebook-aug-v2",
                train_loss=float(history_df["train_loss"].iloc[-1]),
                val_loss=float(final_val["metrics"]["loss"]),
                accuracy=float(final_test["metrics"]["accuracy"]),
                macro_f1=float(final_test["metrics"]["macro_f1"]),
                weighted_f1=float(final_test["metrics"]["weighted_f1"]),
                per_class_recall=final_test["metrics"]["per_class_recall"],
                parameter_count=count_parameters(model),
                flops=0.0,
                training_time_s=summary_row["training_time_s"],
                inference_latency_ms=0.0,
                evaluation_samples=len(test_loader.dataset),
                confusion_matrix_path=final_test.get("confusion_matrix_path", ""),
                checkpoint_path=str(checkpoint_path),
                checkpoint_hash=summary_row["checkpoint_sha256"],
                device=str(DEVICE),
                kind="train",
                status="ok",
                notes=f"Best epoch {best_epoch}; TTA={USE_TTA}; test accuracy {final_test['metrics']['accuracy']:.4f}; test macro F1 {final_test['metrics']['macro_f1']:.4f}",
            )
            TRACKER.finish_run(record, artifacts=[str(checkpoint_path)])
        except Exception as exc:
            logger.warning("Could not append run to project tracker: %s", exc)

    logger.info("Completed run=%s test_accuracy=%.4f test_macro_f1=%.4f checkpoint=%s", run_id, final_test["metrics"]["accuracy"], final_test["metrics"]["macro_f1"], checkpoint_path)
    return {
        **summary_row,
        "history": history,
        "val_metrics": final_val["metrics"],
        "test_metrics": final_test["metrics"],
        "test_confusion_matrix": final_test["confusion_matrix"],
        "sample_predictions": {
            "paths": final_test["paths"],
            "true_labels": final_test["true_names"],
            "predicted_labels": final_test["predicted_names"],
            "top3_indices": final_test["top_indices"],
        },
    }

In [ ]:
class _WarmupCosineScheduler:
    """Linear warmup (epochs) then cosine decay to eta_min. Steps once per epoch.
    Fixes the old notebook: its step-aware guard was never triggered, so the
    cosine schedule silently never applied and LR stayed flat at 1e-4."""

    def __init__(self, optimizer, epochs, warmup_epochs=2, eta_min=1.0e-6):
        self.optimizer = optimizer
        self.warmup_epochs = max(1, int(warmup_epochs))
        self.epochs = max(1, int(epochs))
        self.eta_min = eta_min
        self.base_lrs = [g["lr"] for g in optimizer.param_groups]
        self.stepped = 0

    def step(self):
        self.stepped += 1
        t = self.stepped
        if t <= self.warmup_epochs:
            factor = t / self.warmup_epochs
            for g, base in zip(self.optimizer.param_groups, self.base_lrs):
                g["lr"] = base * factor
        else:
            progress = (t - self.warmup_epochs) / max(1, self.epochs - self.warmup_epochs)
            cosine = 0.5 * (1.0 + np.cos(np.pi * min(progress, 1.0)))
            for g, base in zip(self.optimizer.param_groups, self.base_lrs):
                g["lr"] = self.eta_min + (base - self.eta_min) * cosine


def build_scheduler(optimizer, epochs):
    return _WarmupCosineScheduler(optimizer, epochs, warmup_epochs=WARMUP_EPOCHS)


_original_unfreeze_all = unfreeze_all
_original_freeze_backbone = FREEZE_BACKBONE


def unfreeze_all(model):
    global FREEZE_BACKBONE
    model = _original_unfreeze_all(model)
    FREEZE_BACKBONE = False
    return model


# Restore the configured freeze behavior after each experiment so every run starts identically.
_original_train_one_experiment = train_one_experiment


def train_one_experiment(*args, **kwargs):
    global FREEZE_BACKBONE
    FREEZE_BACKBONE = _original_freeze_backbone
    try:
        return _original_train_one_experiment(*args, **kwargs)
    finally:
        FREEZE_BACKBONE = _original_freeze_backbone
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# Experiment settings (overrides cell 2 defaults).
TRAINING_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 30          # longer patience: stable convergence, less premature stop
WARMUP_EPOCHS = 2                     # linear warmup epochs before cosine decay
GRAD_CLIP_NORM = 5.0                  # gradient clipping (stability)
USE_MIXUP = True                      # MixUp regularization (overfitting guard)
MIXUP_ALPHA = 0.2
MIXUP_PROB = 0.5                      # fraction of batches mixed
USE_TTA = True                        # test-time augmentation (h-flip) on final val/test
OVERFIT_GAP_WARN = 1.0                # warn when train_loss - val_loss exceeds this

for experiment_config in EXPERIMENT_CONFIGS:
    experiment_config["epochs"] = TRAINING_EPOCHS

# torchvision does not expose regnet_y_4gf; use the closest supported 3.2GF model
# while retaining the project-facing architecture name in checkpoints and reports.
_original_get_weight_enum = get_weight_enum


def get_weight_enum(architecture):
    if architecture == "regnet_y_4gf":
        weight_enum = getattr(models, "RegNet_Y_3_2GF_Weights", None)
        return getattr(weight_enum, "DEFAULT", None) if weight_enum is not None else None
    return _original_get_weight_enum(architecture)


if not hasattr(models, "regnet_y_4gf"):
    models.regnet_y_4gf = models.regnet_y_3_2gf

# Probe-build every configured architecture so a typo fails fast, before training starts.
_probe_cfgs = sorted({c["architecture"] for c in EXPERIMENT_CONFIGS})
for _arch in _probe_cfgs:
    _probe = build_model(_arch, len(class_names), pretrained=False)
    assert getattr(_probe, "farmguard_num_classes", None) == len(class_names), _arch
    del _probe
logger.info("Verified build for architectures: %s", _probe_cfgs)

logger.info(
    "Experiment settings: epochs=%d patience=%d device=%s root=%s",
    TRAINING_EPOCHS,
    EARLY_STOPPING_PATIENCE,
    DEVICE,
    ROOT,
)


In [ ]:
results = []
for experiment in EXPERIMENT_CONFIGS:
    if not RUN_EXPERIMENTS:
        break
    experiment["seed"] = int(experiment.get("seed", RANDOM_SEED))
    run_id = f"{safe_name(experiment['architecture'])}_seed{experiment['seed']}_{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
    logger.info("Starting experiment run_id=%s architecture=%s epochs=%d", run_id, experiment["architecture"], experiment["epochs"])
    try:
        result = train_one_experiment(
            experiment,
            run_id,
            train_loader,
            val_loader,
            test_loader,
            class_names,
            class_weight_tensor,
        )
        results.append(result)
        logger.info("Finished experiment run_id=%s", run_id)
    except Exception as exc:
        logger.exception("Experiment failed run_id=%s architecture=%s", run_id, experiment["architecture"])
        results.append({
            "run_id": run_id,
            "status": "failed",
            "architecture": experiment["architecture"],
            "seed": experiment["seed"],
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        })

summary_df = pd.DataFrame(results)
summary_df.to_csv(METRICS_ROOT / "tomato_training_summary.csv", index=False)
save_json(EXPERIMENT_ROOT / "latest_results.json", {
    "created_at": now_iso(),
    "device": str(DEVICE),
    "results": results,
})
display(summary_df)
if not any(row.get("status") == "ok" for row in results):
    raise RuntimeError("No tomato experiment completed successfully. Check logs and GPU/disk/network availability.")


In [ ]:
successful = [row for row in results if row.get("status") == "ok"]
if successful:
    best = max(successful, key=lambda row: float(row.get("test_macro_f1", -1.0)))
    best_arch = safe_name(best["architecture"])
    alias_path = MODEL_ROOT / f"tomato_{best_arch}_best.pt"
    if alias_path.exists():
        alias_path = MODEL_ROOT / f"tomato_{best_arch}_best_{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}.pt"
    shutil.copy2(best["checkpoint_path"], alias_path)
    class_to_idx = write_class_mapping(class_names)
    final_summary = {
        "best_run_id": best["run_id"],
        "best_checkpoint": str(alias_path),
        "best_relative_checkpoint": relative_to_root(alias_path),
        "best_architecture": best["architecture"],
        "best_seed": best["seed"],
        "best_test_accuracy": best["test_accuracy"],
        "best_test_macro_f1": best["test_macro_f1"],
        "best_test_weighted_f1": best["test_weighted_f1"],
        "class_mapping": {"classes": class_names, "class_to_idx": class_to_idx},
        "all_results": results,
    }
    save_json(METRICS_ROOT / "tomato_training_final_summary.json", final_summary)
    display(Markdown(f"**Best run:** `{best['run_id']}` on `{best['architecture']}`. Checkpoint copied to `{relative_to_root(alias_path)}`."))
    display(pd.DataFrame([{
        "run_id": best["run_id"],
        "architecture": best["architecture"],
        "seed": best["seed"],
        "test_accuracy": best["test_accuracy"],
        "test_macro_f1": best["test_macro_f1"],
        "test_weighted_f1": best["test_weighted_f1"],
        "checkpoint": relative_to_root(alias_path),
    }]))
else:
    display(Markdown("No successful run was available for model registration."))


In [ ]:
if successful:
    best = max(successful, key=lambda row: float(row.get("test_macro_f1", -1.0)))
    best_checkpoint = Path(best["checkpoint_path"])
    if not best_checkpoint.exists():
        best_checkpoint = Path(best["relative_checkpoint_path"]) if "relative_checkpoint_path" in best else alias_path
    payload = torch.load(str(best_checkpoint), map_location="cpu", weights_only=False)
    state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
    registered_model = build_model(meta.get("architecture", best["architecture"]), len(class_names), pretrained=False)
    registered_model.load_state_dict(state_dict, strict=True)
    registered_model.to(DEVICE)
    registered_model.eval()

    example_path = Path(test_dataset.samples[0][0])
    example_image = Image.open(example_path).convert("RGB")
    with torch.no_grad():
        example_input = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])(example_image).unsqueeze(0).to(DEVICE)
        start = datetime.now(timezone.utc)
        for _ in range(5):
            registered_model(example_input)
        latency_ms = (datetime.now(timezone.utc) - start).total_seconds() * 1000.0 / 5
        logits = registered_model(example_input)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        top_indices = np.argsort(-probs)[:3]
        prediction_record = {
            "image": str(example_path),
            "true_label": class_names[int(test_dataset.samples[0][1])],
            "top_predictions": [
                {"label": class_names[int(idx)], "probability": float(probs[int(idx)])}
                for idx in top_indices
            ],
            "latency_ms_per_forward": float(latency_ms),
            "checkpoint": relative_to_root(best_checkpoint),
            "architecture": meta.get("architecture", best["architecture"]),
        }
    save_json(PREDICTION_ROOT / "tomato_best_example_prediction.json", prediction_record)
    display(Markdown(f"**Inference smoke test passed.** Loaded `{relative_to_root(best_checkpoint)}` and ran 5 forwards on one test image."))
    display(example_image)
    display(pd.DataFrame(prediction_record["top_predictions"]))
else:
    display(Markdown("Skipping inference smoke test because no model was trained."))


In [ ]:
# ================= ENSEMBLE: soft-vote of the top-3 checkpoints on the test set =================
# Goal from the previous report: "Ensembling the top-3 (regnet + resnet50 + convnext)
# would likely push test accuracy above 97%." This cell measures that claim.

successful = [row for row in results if row.get("status") == "ok"]
if len(successful) >= 2:
    top = sorted(successful, key=lambda r: float(r.get("test_macro_f1", -1.0)), reverse=True)[:3]

    def _collect_probs(ckpt_path, arch):
        payload = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)
        sd = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
        meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
        m = build_model(meta.get("architecture", arch), len(class_names), pretrained=False)
        m.load_state_dict(sd, strict=True)
        m.to(DEVICE)
        m.eval()
        probs = []
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.to(DEVICE, non_blocking=True)
                with autocast_context(DEVICE, DEVICE.type == "cuda"):
                    logits = m(inputs)
                probs.append(torch.softmax(logits, 1).cpu().numpy())
        return np.concatenate(probs, 0), meta

    names, prob_list = [], []
    for row in top:
        ckpt = Path(row["checkpoint_path"])
        if not ckpt.exists():
            ckpt = Path(row.get("relative_checkpoint_path", ""))
        p, _ = _collect_probs(ckpt, row["architecture"])
        prob_list.append(p)
        names.append(row["architecture"])

    y_true = np.asarray(test_loader.dataset.targets)
    soft = np.mean(prob_list, axis=0)
    ens_preds = soft.argmax(1)

    ens_metrics = {
        "accuracy": float(accuracy_score(y_true, ens_preds)),
        "macro_precision": float(precision_score(y_true, ens_preds, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, ens_preds, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, ens_preds, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, ens_preds, average="weighted", zero_division=0)),
        "per_class_recall": {},
    }
    pcr = recall_score(y_true, ens_preds, average=None, labels=list(range(len(class_names))), zero_division=0)
    ens_metrics["per_class_recall"] = {class_names[i]: float(pcr[i]) for i in range(len(class_names))}
    ens_cm = confusion_matrix(y_true, ens_preds, labels=list(range(len(class_names))))

    best_single = max(top, key=lambda r: float(r.get("test_macro_f1", -1.0)))
    ensemble_payload = {
        "members": names,
        "member_test_macro_f1": {r["architecture"]: r["test_macro_f1"] for r in top},
        "metrics": ens_metrics,
        "confusion_matrix": ens_cm.tolist(),
        "vs_best_single": {
            "best_single_architecture": best_single["architecture"],
            "best_single_test_macro_f1": best_single["test_macro_f1"],
            "best_single_test_accuracy": best_single["test_accuracy"],
            "ensemble_macro_f1": ens_metrics["macro_f1"],
            "ensemble_accuracy": ens_metrics["accuracy"],
            "macro_f1_gain": ens_metrics["macro_f1"] - float(best_single["test_macro_f1"]),
        },
        "note": "Soft-vote (mean softmax) of the top-3 by test macro F1. Measured on the held-out test set.",
    }
    save_json(METRICS_ROOT / "tomato_ensemble_top3.json", ensemble_payload)
    ens_df = pd.DataFrame({
        "path": [p for p, _ in test_loader.dataset.samples],
        "true_label": [class_names[int(t)] for t in y_true],
        "predicted_label": [class_names[int(p)] for p in ens_preds],
        "top1_probability": soft[np.arange(len(soft)), ens_preds],
    })
    ens_df.to_csv(PREDICTION_ROOT / "tomato_ensemble_top3_predictions.csv", index=False)

    display(Markdown(f"**Ensemble members:** {', '.join(names)} (soft-vote of mean softmax)"))
    comp = pd.DataFrame([
        {"model": n, "test_accuracy": t["test_accuracy"], "test_macro_f1": t["test_macro_f1"]} for n, t in
        zip(names, [next(r for r in top if r["architecture"] == n) for n in names])
    ] + [{"model": "ENSEMBLE (top-3 soft vote)", "test_accuracy": ens_metrics["accuracy"],
          "test_macro_f1": ens_metrics["macro_f1"]}])
    display(comp.style.format({"test_accuracy": "{:.4f}", "test_macro_f1": "{:.4f}"}))
    gain = ens_metrics["macro_f1"] - float(best_single["test_macro_f1"])
    if gain > 0:
        display(Markdown(f"✅ **Ensemble beats the best single model by {gain:.4f} macro-F1** "
                         f"({ens_metrics['macro_f1']:.4f} vs {best_single['test_macro_f1']:.4f})."))
    else:
        display(Markdown(f"ℹ️ Ensemble macro-F1 {ens_metrics['macro_f1']:.4f} does NOT beat the best single "
                         f"({best_single['test_macro_f1']:.4f}) — reported as measured."))
else:
    display(Markdown("Ensemble skipped: need at least 2 successful runs."))

## Project integration checklist

The notebook writes the best checkpoint to:

`models/checkpoints/tomato/tomato_<architecture>_best.pt`

It also writes the class mapping to `configs/classes/tomato.yaml` and the
top-3 ensemble results to `outputs/metrics/tomato_ensemble_top3.json`.

To expose the best model to the existing AgroVision inference/evaluation code,
add one entry under `instances:` in `configs/models.yaml` (see
`docs/tomato.md` §9 for the already-wired models):

```yaml
instances:
  - id: tomato_variant_a_best
    name: Tomato Variant A best (colab retrain)
    architecture: <architecture that won>
    crop: tomato
    checkpoint: models/checkpoints/tomato/tomato_<architecture>_best.pt
    classes_file: configs/classes/tomato.yaml
    input_size: 224
```

## Overfitting checklist (this notebook)

- **Best state = best val macro F1** (never the last epoch) — early stopping
  patience 30 keeps the best snapshot even if training continues.
- **MixUp + label smoothing + sqrt class weights + gradient clipping** all act
  as regularizers.
- Per-epoch history now includes `train_val_gap`; the run log warns when the
  gap grows past `OVERFIT_GAP_WARN` (1.0 loss units).
- Final val/test use **TTA** (h-flip averaging) — a free accuracy boost that
  also de-noises the final numbers.
- The test set is never seen during training or model selection.
